# Task 3: A/B Hypothesis Testing

This notebook tests four insurance-risk hypotheses using explicit control/test segmentation. Claim frequency is treated as a categorical KPI and tested with chi-squared tests. Claim severity and margin are numerical KPIs and tested with Welch independent t-tests.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

sys.path.append(str(Path.cwd().parent / "src"))
from data_loader import load_insurance_data, add_loss_ratio

df = add_loss_ratio(load_insurance_data(Path.cwd().parent / "data" / "raw" / "insurance_data.csv"))
df["ClaimOccurred"] = (df["TotalClaims"] > 0).astype(int)
df["ClaimSeverity"] = df["TotalClaims"].where(df["TotalClaims"] > 0)
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]
df["ZipCodeDistrict"] = df["ZipCode"].astype(str).str[:2]
df[["Province", "ZipCode", "ZipCodeDistrict", "Gender", "TotalPremium", "TotalClaims", "ClaimOccurred", "ClaimSeverity", "Margin"]].head()

,Province,ZipCode,ZipCodeDistrict,Gender,TotalPremium,TotalClaims,ClaimOccurred,ClaimSeverity,Margin
0,Western Cape,1863,18,Female,2106.60,0.00,0,NaN,2106.60
1,North West,4055,40,Female,2443.31,0.00,0,NaN,2443.31
2,Free State,6133,61,Female,2041.02,3106.97,1,3106.97,-1065.95
3,Eastern Cape,3682,36,Male,1087.96,0.00,0,NaN,1087.96
4,Gauteng,6278,62,Male,2594.12,997.47,1,997.47,1596.65


## Helper Functions

Each test records Group A baseline, Group B comparison, KPI, statistical test, p-value, and the decision against alpha = 0.05.

In [2]:
ALPHA = 0.05
results = []

def decision(p_value):
    return "Reject H0" if p_value < ALPHA else "Fail to reject H0"

def add_result(hypothesis, group_a, group_b, kpi, test, p_value, statistic):
    results.append({
        "Hypothesis": hypothesis,
        "Group A baseline": group_a,
        "Group B comparison": group_b,
        "KPI": kpi,
        "Statistical test": test,
        "Statistic": round(float(statistic), 4),
        "p-value": round(float(p_value), 6),
        "Decision": decision(float(p_value)),
    })

## H1: No Risk Differences Across Provinces

- H0: Claim frequency is independent of province.
- Group A baseline: province with the largest policy count.
- Group B comparison: all other provinces in the province contingency table.
- KPI: Claim Frequency (`ClaimOccurred`).
- Test: chi-squared test of independence.

In [3]:
province_counts = df["Province"].value_counts()
baseline_province = province_counts.idxmax()
province_table = pd.crosstab(df["Province"], df["ClaimOccurred"])
chi2, p_value, dof, expected = stats.chi2_contingency(province_table)
display(province_table)
add_result(
    "No risk differences across provinces",
    baseline_province,
    "All other provinces",
    "Claim Frequency",
    "Chi-squared test",
    p_value,
    chi2,
)

ClaimOccurred,0,1
Province,,
Eastern Cape,616,401
Free State,381,287
Gauteng,1518,1029
KwaZulu-Natal,974,643
Limpopo,490,298
Mpumalanga,424,240
North West,350,234
Northern Cape,233,144
Western Cape,1006,732


## H2: No Risk Differences Between ZIP Codes

The raw ZIP codes are granular, so ZIP-code districts are created from the first two ZIP digits to preserve geographic ZIP structure while ensuring enough observations per group.

- H0: Claim frequency is independent of ZIP-code district.
- Group A baseline: most common ZIP-code district.
- Group B comparison: all other ZIP-code districts.
- KPI: Claim Frequency (`ClaimOccurred`).
- Test: chi-squared test of independence.

In [4]:
zip_counts = df["ZipCodeDistrict"].value_counts()
baseline_zip = zip_counts.idxmax()
zip_table = pd.crosstab(df["ZipCodeDistrict"], df["ClaimOccurred"])
chi2, p_value, dof, expected = stats.chi2_contingency(zip_table)
display(zip_table.head(12))
add_result(
    "No risk differences between ZIP codes",
    f"ZIP district {baseline_zip}",
    "All other ZIP districts",
    "Claim Frequency",
    "Chi-squared test",
    p_value,
    chi2,
)

ClaimOccurred,0,1
ZipCodeDistrict,,
10,77,46
11,74,52
12,55,49
13,64,42
14,76,37
15,71,43
16,51,37
17,75,33
18,60,39


## H3: No Significant Margin Differences Between ZIP Codes

- H0: Mean margin is the same between the baseline ZIP-code district and comparison ZIP-code district.
- Group A baseline: most common ZIP-code district.
- Group B comparison: second most common ZIP-code district.
- KPI: Margin (`TotalPremium - TotalClaims`).
- Test: Welch independent t-test.

In [5]:
comparison_zip = zip_counts.index[1]
margin_a = df.loc[df["ZipCodeDistrict"] == baseline_zip, "Margin"].dropna()
margin_b = df.loc[df["ZipCodeDistrict"] == comparison_zip, "Margin"].dropna()
t_stat, p_value = stats.ttest_ind(margin_a, margin_b, equal_var=False)
summary = pd.DataFrame({
    "ZipCodeDistrict": [baseline_zip, comparison_zip],
    "n": [len(margin_a), len(margin_b)],
    "MeanMargin": [margin_a.mean(), margin_b.mean()],
    "StdMargin": [margin_a.std(), margin_b.std()],
})
display(summary)
add_result(
    "No significant margin differences between ZIP codes",
    f"ZIP district {baseline_zip}",
    f"ZIP district {comparison_zip}",
    "Margin",
    "Welch t-test",
    p_value,
    t_stat,
)

,ZipCodeDistrict,n,MeanMargin,StdMargin
0,97,133,-379.364812,6398.512009
1,64,131,499.036641,3267.405024


## H4: No Significant Risk Difference Between Women and Men

- H0: Claim severity is the same for women and men.
- Group A baseline: women.
- Group B comparison: men.
- KPI: Claim Severity (`TotalClaims` where claims occurred).
- Test: Welch independent t-test.

In [6]:
female_severity = df.loc[(df["Gender"] == "Female") & (df["TotalClaims"] > 0), "TotalClaims"].dropna()
male_severity = df.loc[(df["Gender"] == "Male") & (df["TotalClaims"] > 0), "TotalClaims"].dropna()
t_stat, p_value = stats.ttest_ind(female_severity, male_severity, equal_var=False)
gender_summary = pd.DataFrame({
    "Gender": ["Female", "Male"],
    "n_claims": [len(female_severity), len(male_severity)],
    "MeanSeverity": [female_severity.mean(), male_severity.mean()],
    "StdSeverity": [female_severity.std(), male_severity.std()],
})
display(gender_summary)
add_result(
    "No significant risk difference between women and men",
    "Female",
    "Male",
    "Claim Severity",
    "Welch t-test",
    p_value,
    t_stat,
)

,Gender,n_claims,MeanSeverity,StdSeverity
0,Female,1922,3983.173361,4621.734453
1,Male,1962,4405.604985,6273.386238


## Results Summary

The table below summarizes all four null hypotheses, the statistical test used, p-value, and decision.

In [7]:
results_df = pd.DataFrame(results)
display(results_df)
reports_dir = Path.cwd().parent / "reports"
reports_dir.mkdir(exist_ok=True)
results_df.to_csv(reports_dir / "task3_hypothesis_results.csv", index=False)

,Hypothesis,Group A baseline,Group B comparison,KPI,Statistical test,Statistic,p-value,Decision
0,No risk differences across provinces,Gauteng,All other provinces,Claim Frequency,Chi-squared test,12.1914,0.142866,Fail to reject H0
1,No risk differences between ZIP codes,ZIP district 97,All other ZIP districts,Claim Frequency,Chi-squared test,100.2740,0.194549,Fail to reject H0
2,No significant margin differences between ZIP ...,ZIP district 97,ZIP district 64,Margin,Welch t-test,-1.4078,0.160768,Fail to reject H0
3,No significant risk difference between women a...,Female,Male,Claim Severity,Welch t-test,-2.3926,0.016780,Reject H0
